# Exercise 3 — Overlay ZTF / AGN on the GW sky map

Put the **ZTF19abanrhr** / AGN J124942.3+344929 position on the **GW190521** PE sky map — a first look at spatial coincidence for the ZTF–GW cross-correlation project.

```bash
uv run jupyter notebook notebooks/exercises/03_overlay_ztf_on_skymap.ipynb
```

Requires Exercises 1–2 data (or this notebook will fetch both).

In [ ]:
from pathlib import Path
import sys

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
from ligo.skymap.io import read_sky_map
from ligo.skymap.postprocess import find_greedy_credible_levels


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "fetch_data.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root (scripts/fetch_data.py)")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "scripts"))
from fetch_data import ZTF19ABANRHR, fetch_skymaps_gwosc, fetch_ztf_irsa

paths = fetch_skymaps_gwosc(["GW190521"])
skymap_path = paths[0] if paths else ROOT / "data" / "skymaps" / "GW190521_skymap.fits"
fetch_ztf_irsa(object_name="ZTF19abanrhr")
print("Sky map:", skymap_path)

In [ ]:
prob, meta = read_sky_map(str(skymap_path), nest=True)
credible = find_greedy_credible_levels(prob)

ra = ZTF19ABANRHR["ra"]
dec = ZTF19ABANRHR["dec"]
theta = np.deg2rad(90.0 - dec)
phi = np.deg2rad(ra)
nside = hp.npix2nside(len(prob))
ipix = hp.ang2pix(nside, theta, phi, nest=True)
p_at = prob[ipix]
cl_at = credible[ipix]

fig = plt.figure(figsize=(8, 4.2))
hp.mollview(
    prob,
    nest=True,
    title="GW190521 sky map + ZTF19abanrhr",
    unit="probability density",
    cmap="cylon",
    hold=True,
    fig=fig.number,
)
hp.graticule(dpar=30, dmer=60)
hp.contour(credible, nest=True, levels=[0.5, 0.9], colors=["0.85", "white"], linewidths=[0.8, 1.2], hold=True)
hp.projscatter(theta, phi, lonlat=False, s=60, marker="*", c="cyan", edgecolors="k", linewidths=0.6, label="ZTF19abanrhr")
plt.legend(loc="lower left")
plt.show()

print(f"AGN / ZTF position: RA={ra:.6f}°, Dec={dec:.6f}°")
print(f"Pixel probability density: {p_at:.3e}")
print(f"Smallest credible level enclosing this pixel: {100 * cl_at:.1f}%")
print("(Values < 90% mean the source lies inside the 90% region.)")

### Try next

- Zoom with `hp.gnomview` centered on the AGN.
- Estimate chance coincidence given the 90% area and an AGN surface density.
- Repeat for another well-localized BBH from GWTC.